In [ ]:
import numpy as np
import xarray as xr

from frequensolve.project import Project
from frequensolve.seismic import Acquisition, ReceiverFiber, ReceiverNode
from frequensolve.seismic.layered_model import LayeredModel
from frequensolve.seismic.wavelet import RickerWavelet
from frequensolve.mesh import BoundaryCondition, BoundaryConditionManager
from frequensolve.simulation import (
    Discretization,
    FrequencyDomainJob,
    OutputManager,
    ParaviewOutput,
    SolverConfig,
)

project_path = "./scratch/ex_04/"
project = Project(
    name = "project",
    pretty_name = "DAS Simulation",
    path = project_path,
)
 
sim = project.new_simulation(name = "simulation_1", physics = "coupled", dimension = 2)

# --- Model ---
# New 2D layered model over interval x in [0,1]
model = LayeredModel(dimension = 2, x_limits = [0.0, 1.0])

model.add_surface(name = "top", z = 0.0)
model.add_layer(
    name = "layer_1",
    properties = { 
        "Vp" : 1.0,
        "Qp" : 100.0,
        "Vs" : 0.5,
        "Qs" : 100.0,
        "Rho": 2.0
    }
)

model.add_surface(
    name = "interface", 
    z = xr.DataArray(
        [0.2, 0.35], 
        dims = ["x"], 
        coords = {"x" : [0.0, 1.0]}
    )
)

model.add_layer(                    # Layer 2 
    name = "layer_2",
    properties = { 
        "Vp" : 4.0,
        "Qp" : 100.0,
        "Vs" : 2.0,
        "Qs" : 100.0,
        "Rho": 2.0 
    }
)
model.add_surface(name = "bottom", z = 0.5)

# --- Mesh ---
mesh = model.hex_mesh_generator(n = [4, 4])
sim += mesh
sim.mesh.set_adapt(
    min_epw = 1.5,
    adapt_sources = 2
)

# --- Boundary Conditions ---
BCs = BoundaryConditionManager(label_type = "geometric")
BCs += BoundaryCondition(
    name = "free_surface",
    kind = "impedance",
    boundaries = ["z_min"]
)
BCs += BoundaryCondition(
    name = "pml",
    kind = "pml",
    boundaries = ["x_min", "x_max", "z_max"],
    pml_wavelengths = 1.0,
    pml_exponent = 3.0,
    pml_constant = 3.0
)
sim += BCs

In [ ]:

# --- Acquisition ---
acq = Acquisition()

# define source
acq.add_source_group(kind = "vector",
                     coords = [[0.5, 0.0]],
                     direction = [0.0, 1.0])

# surface geophones
geophone = ReceiverNode(name = "geophone")
geophone.add_component("u_z","velocity",[0.0, 1.0])

coords = [ [x, 0.1] for x in np.linspace(0.0, 1.0, 1001)]
acq.add_receiver_group(
    name = "surface_geophones",
    device = geophone,
    coords = coords,
    frame = "reference"
)

# downhole DAS
das = ReceiverFiber(
    name = "fiber",
    L_gauge = 0.005,
    n_gauge = 251,
    radius = 0.001,
    pitch = 0.0025,
)
das.add_component("eps_tt", "strain")


# n_turns = 5
# r0 = 0.025
# rf = 0.4 
# z0 = 0.01
# zf = 0.50
# n_pts = 51
# coords = []
# for i, t in enumerate(np.linspace(0.0, 1.0, n_pts)):
#     theta = n_turns * 2 * np.pi * t
#     r = r0 + (rf - r0) * t
#     x = 0.5 + r * np.cos(theta)
#     z = z0 + (zf - z0) * t + 0.01*np.sin(10*theta)  # add some small oscillation to z
#     coords.append([x, z])

coords = [ [0.25, z] for z in np.linspace(0.0, 0.5, 101)]
acq.add_receiver_group(
    name = "well_das",
    device = das,
    coords = coords,
    frame = "physical",
    post_process = True,
)
sim += acq

model.plot("Vp", acquisition = acq)
sim += model

# --- Discretization ---
sim += Discretization(order = 4)
sim += SolverConfig(tolerance = 1.0e-4, grids = 3, kwargs = {"ptol": 1.e-10})

out = OutputManager()
out += ParaviewOutput(
    name = "das", 
    fields = ["velocity", "velocity", "pressure"],
    properties = ["Vp", "Subdomain"],
    upscale = 1,
)
sim += out

project.save()

In [ ]:
from frequensolve.orchestrator.sites.local import LocalSite

# --- Connect to site ---
site = LocalSite()
site.sync(project)

s = 0.1

# Create and submit frequency domain job
job = TimeDomainJob(
    name = "td_job",
    simulation = sim,
    s_laplace = -s,
    f_min = 1.0,
    f_max = 20.0,
    T_max = 4.0
)

fd_job = FrequencyDomainJob(
    name = "freq",
    simulation = sim,
    f_list = [15.0 - 0.1j],
)
site.run(job)
trace_db = site.fetch_traces(job, upscale = 4)

wavelet = RickerWavelet(f = 6, center = 0.125)
wavelet.plot(f_max = 20.0)

print(trace_db)
for group in trace_db.groups:
    for shot in trace_db.shots(group):
        for comp in trace_db.components(group):
            td = trace_db.td(group, comp, shot, wavelet, upscale = 4)
            plot = td.plot(
                vmin = -.1, 
                vmax = .1, 
                y = "time", 
                yincrease = False,
                cmap = "gray"
            )
            exp = xr.DataArray(
                np.exp(2*np.pi*s*td.coords["time"]),
                dims = ["time"],
                coords = {"time": td.coords["time"]},
            )
            td *= exp

            # fd = trace_db.fd(group, comp, shot, wavelet)
            # plot = fd.real.plot(
            #     vmin = -10, 
            #     vmax = 10, 
            #     y = "frequency", 
            #     yincrease = False,
            #     cmap = "gray"
            # )
            plt.show()